# 03 · Clustering — Ward dendrogram + t-SNE

Two methodological choices that anchor this notebook:

1. **Ward agglomerative clustering** — yields a *full dendrogram*; we cut at k=5. The dendrogram itself is later consumed by Blender to drive a "branch unfolding" reveal animation, so the clustering algorithm is not just an analytical tool but also the animation's source of structure.
2. **t-SNE** — local-neighbourhood-preserving manifold learning, used purely for the 2-D scatter plot of the cluster assignments.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, json, logging
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(name)s | %(message)s')

from src.config import CLUSTERS_DIR, FRAGMENT_PARAMS_DIR, PROCESSED_DIR, VECTORS_DIR, N_CLUSTERS
from src.vectorize.sbert import SBertArtefact

corpus = pd.read_parquet(PROCESSED_DIR / 'corpus.parquet')
sbert = SBertArtefact.load(VECTORS_DIR / 'sbert.npz')
print('sbert  :', sbert.embeddings.shape)

## 3.1 Align SBERT vectors with the corpus order
SBERT is the primary signal that feeds Ward / t-SNE / 3D parameter generation.

In [ ]:
id_to_sbert = {i: v for i, v in zip(sbert.ids, sbert.embeddings)}
txt_dim = sbert.embeddings.shape[1]

rows, ordered_ids = [], []
for _, row in corpus.iterrows():
    rid = row['id']
    rows.append(id_to_sbert.get(rid, np.zeros(txt_dim, dtype=np.float32)))
    ordered_ids.append(rid)

fused = np.vstack(rows).astype(np.float32)
norms = np.linalg.norm(fused, axis=1, keepdims=True)
fused = fused / np.maximum(norms, 1e-9)
print('matrix:', fused.shape)

## 3.2 Ward hierarchical clustering — full dendrogram + cut at k=5

In [ ]:
from src.ml import run_ward, fragments_from_clusters
ward = run_ward(fused, k=N_CLUSTERS, embedding_kind='sbert')
print('ward sizes:', ward.cluster_sizes, ' silhouette:', round(ward.silhouette, 4))

np.save(CLUSTERS_DIR / 'ward_labels.npy', ward.labels)
np.save(CLUSTERS_DIR / 'ward_linkage.npy', ward.linkage_matrix)

## 3.3 t-SNE — 2-D projection for visual inspection (notebook 04 plots it).

In [ ]:
from src.ml import project_tsne
tsne = project_tsne(fused)
np.save(CLUSTERS_DIR / 'tsne_2d.npy', tsne)
print('t-SNE:', tsne.shape)

## 3.4 Bridge to 3D — write `clusters.json`
Materialise 5 × 4 = 20 fragments. Each fragment carries a 6-D parameter vector mapped from the cluster centroid through a deterministic random projection.

In [ ]:
json_path = fragments_from_clusters(ward)
print('clusters.json →', json_path)

## 3.5 Inspect a few fragments to confirm the JSON shape

In [ ]:
payload = json.loads(json_path.read_text(encoding='utf-8'))
for f in payload['fragments'][:3]:
    print(f['fragment_id'], f['archetype'], '→', f['params'])

In [ ]:
print('fragments:', len(payload['fragments']),
      ' | archetypes:', sorted({f['archetype'] for f in payload['fragments']}))